# 한국어 음성 인식, 같은 Whisper tiny를 네 가지 구현으로 실행하기

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/korean_asr_runtime_benchmark.ipynb)

Colab CPU에서 FLEURS 한국어 테스트 세 발화에 대한 어절 오류율(WER), 공백을 뺀 글자 오류율(CER), 실시간 배율(RTF)을 측정합니다. 모델을 불러온 뒤 세 발화를 한 번 처리해 워밍업하고 다시 다섯 번 잽니다. 오디오 디코딩과 가중치 다운로드는 타이머 밖에 둡니다. `CHART_LANG`은 ko, en, both 중에서 고릅니다.

음원과 정답 전사는 Google FLEURS의 CC BY 4.0 자료입니다. 정규화한 전사로 오류를 집계하며 저장소에 음원을 담지 않습니다.

In [ ]:
%pip -q install openai-whisper==20250625 faster-whisper==1.2.1 pywhispercpp==1.5.1 "numba>=0.65"

## Environment

In [ ]:
import gc
import io
import json
import os
import platform
import statistics
import time
import unicodedata
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pyarrow.parquet as pq
import soundfile as sf
import torch
from huggingface_hub import hf_hub_download, snapshot_download

THREADS = min(2, len(os.sched_getaffinity(0)))
torch.set_num_threads(THREADS)
torch.manual_seed(17)
print(platform.python_version(), platform.machine(), "threads", THREADS)
for package in ("torch", "transformers", "openai-whisper", "faster-whisper",
                "ctranslate2", "pywhispercpp", "numpy"):
    print(package, version(package))

## The first three distinct prompt IDs in a pinned FLEURS test shard

In [ ]:
# FLEURS by Google, CC BY 4.0. Audio and reference text are fetched at run time.
# https://huggingface.co/datasets/google/fleurs
shard = hf_hub_download("google/fleurs", "ko_kr/test/0000.parquet", repo_type="dataset",
                        revision="168de341b3db6859a9bac1c50a2ef5e3b47647e0")
clips = []
seen_ids = set()
for batch in pq.ParquetFile(shard).iter_batches(batch_size=8, columns=["id", "transcription", "audio"]):
    for row in batch.to_pylist():
        if row["id"] in seen_ids:
            continue
        audio, rate = sf.read(io.BytesIO(row["audio"]["bytes"]), dtype="float32")
        assert rate == 16000 and audio.ndim == 1 and len(audio) <= 30 * rate
        clips.append((row["id"], audio, row["transcription"]))
        seen_ids.add(row["id"])
        if len(clips) == 3:
            break
    if len(clips) == 3:
        break
assert len(clips) == 3
duration = sum(len(audio) / 16000 for _, audio, _ in clips)
print("Samples:", [(clip_id, len(audio) / 16000) for clip_id, audio, _ in clips])
print("Total audio seconds:", duration)

## Metrics and repeated decoding

In [ ]:
def normalized(text):
    text = unicodedata.normalize("NFC", text).casefold()
    return " ".join("".join(ch for ch in text if not unicodedata.category(ch).startswith("P")).split())

def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, ref in enumerate(reference, 1):
        current = [i]
        for j, hyp in enumerate(hypothesis, 1):
            current.append(min(previous[j] + 1, current[j-1] + 1, previous[j-1] + (ref != hyp)))
        previous = current
    return previous[-1]

def counts(reference, hypothesis):
    ref, hyp = normalized(reference), normalized(hypothesis)
    words = (ref.split(), hyp.split())
    chars = (ref.replace(" ", ""), hyp.replace(" ", ""))
    return (edit_distance(*words), len(words[0]), edit_distance(*chars), len(chars[0]))

assert counts("한 글", "한글") == (2, 2, 0, 2)
results = {}
def benchmark(name, transcribe):
    def once():
        return [transcribe(audio).strip() for _, audio, _ in clips]

    warmup = once()
    elapsed = []
    for _ in range(5):
        start = time.perf_counter()
        texts = once()
        elapsed.append(time.perf_counter() - start)
        assert texts == warmup, f"Transcript changed across repetitions: {name}"
    word_errors, words, char_errors, chars = map(int, np.sum(
        [counts(ref, hyp) for (_, _, ref), hyp in zip(clips, warmup)], axis=0))
    results[name] = dict(word_errors=word_errors, words=words, char_errors=char_errors, chars=chars,
                         wer=100 * word_errors / words, cer=100 * char_errors / chars,
                         seconds=elapsed, rtf=[s / duration for s in elapsed])
    print(name, json.dumps(results[name]))

## OpenAI Whisper (MIT). Load once before starting the clock.

In [ ]:
import whisper

model = whisper.load_model("tiny", device="cpu")
def reference_decode(audio):
    return model.transcribe(audio, language="ko", task="transcribe", temperature=0,
                            without_timestamps=True, fp16=False, condition_on_previous_text=False,
                            verbose=None)["text"]

benchmark("OpenAI", reference_decode)
del model
gc.collect()

## Transformers distribution of Whisper tiny (model card: Apache-2.0).

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

repo = "openai/whisper-tiny"
revision = "169d4a4341b33bc18d8881c4b69c2e104e1cc0af"
processor = WhisperProcessor.from_pretrained(repo, revision=revision)
model = WhisperForConditionalGeneration.from_pretrained(repo, revision=revision).eval()

@torch.inference_mode()
def transformers_decode(audio):
    batch = processor(audio, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
    tokens = model.generate(**batch, language="ko", task="transcribe", do_sample=False,
                            num_beams=1, return_timestamps=False)
    return processor.batch_decode(tokens, skip_special_tokens=True)[0]

benchmark("Transformers", transformers_decode)
del model, processor
gc.collect()

## CTranslate2 distribution from SYSTRAN (MIT).

In [ ]:
from faster_whisper import WhisperModel

directory = snapshot_download("Systran/faster-whisper-tiny",
                              revision="d90ca5fe260221311c53c58e660288d3deb8d356")
model = WhisperModel(directory, device="cpu", compute_type="float32", cpu_threads=THREADS)
def faster_decode(audio):
    segments, _ = model.transcribe(audio, language="ko", task="transcribe", beam_size=1,
                                   best_of=1, temperature=0, vad_filter=False,
                                   condition_on_previous_text=False, without_timestamps=True)
    return "".join(segment.text for segment in segments)

benchmark("faster-whisper", faster_decode)
del model
gc.collect()

## GGML distribution from whisper.cpp (MIT), Python bindings pywhispercpp.

In [ ]:
from pywhispercpp.model import Model

weights = hf_hub_download("ggerganov/whisper.cpp", "ggml-tiny.bin",
                          revision="5359861c739e955e79d9a303bcbc70fb988958b1")
model = Model(weights, params_sampling_strategy=0, language="ko", n_threads=THREADS,
              no_context=True, no_timestamps=True, print_progress=False, print_realtime=False,
              temperature=0.0, temperature_inc=0.0, greedy={"best_of": 1})
def cpp_decode(audio):
    return "".join(s.text for s in model.transcribe(audio))

benchmark("whisper.cpp", cpp_decode)
del model
gc.collect()

## Two views of the same measured data

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font = Path("NanumGothic-Regular.ttf")
if not font.exists():
    urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf", font)
font_manager.fontManager.addfont(str(font))
plt.rcParams.update({"font.family": "NanumGothic", "axes.spines.top": False, "axes.spines.right": False})
requested = os.getenv("CHART_LANG", "ko")
languages = ["ko", "en"] if requested == "both" else [requested]
for lang in languages:
    ko = lang == "ko"
    labels = list(results)
    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    position = np.arange(len(labels))
    for offset, key, color in [(-0.18, "wer", "#316591"), (0.18, "cer", "#ad6734")]:
        bars = ax.bar(position + offset, [v[key] for v in results.values()], 0.36,
                      label=key.upper(), color=color)
        ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set_xticks(position, labels)
    ax.set_ylabel("오류율 (%)" if ko else "Error rate (%)")
    ax.set_ylim(0, max(max(v["wer"], v["cer"]) for v in results.values()) * 1.23)
    ax.set_title("한국어 세 발화의 어절과 글자 오류율" if ko else "Word and character errors across three Korean utterances")
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    ax.legend()
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    for i, row in enumerate(results.values()):
        ratios = row["rtf"]
        mid = statistics.median(ratios)
        ax.plot([min(ratios), max(ratios)], [i, i], lw=3, color="#7194af")
        ax.plot(mid, i, "o", color="#254861")
        ax.annotate(f"{mid:.3f}", (max(ratios), i), xytext=(8, 0),
                    textcoords="offset points", va="center")
    ax.set_yticks(range(len(labels)), labels)
    ax.invert_yaxis()
    ax.set_xlim(0, max(max(v["rtf"]) for v in results.values()) * 1.25)
    ax.set_xlabel("처리 시간 / 음성 길이" if ko else "Processing time / audio duration")
    ax.set_title("실시간 배율, 5회 중앙값과 범위" if ko else "Real-time factor, median and range of five runs")
    ax.grid(axis="x", alpha=0.18)
    plt.show()
    plt.close(fig)